# nano-dsv4.1f — Kaggle TPU v5e-8

Checked-in repo notebook. It prioritizes **HBM safety and a modest 32K softmax** over tokenizer-efficiency experiments. Run it from a checkout of this repository. For a single-file self-contained export, run `python scripts/build_notebook.py` from the repo.

In [ ]:
from pathlib import Path
import os,sys
root=Path.cwd()
for c in (root,root.parent):
    if (c/'pyproject.toml').exists() and (c/'src'/'nano_dsv41f').exists(): root=c; break
else: raise RuntimeError('Repository source not found. Run this notebook from a checkout of nano-dsv4.1f, or generate the self-contained export with scripts/build_notebook.py.')
os.chdir(root); sys.path.insert(0,str(root/'src')); print('repo root:',root)

In [ ]:
import jax, jax.numpy as jnp
print('JAX:',jax.__version__)
print('devices:',jax.devices())
if not jax.devices() or jax.devices()[0].platform!='tpu': raise RuntimeError('Select TPU in Kaggle Settings > Accelerator first.')

In [ ]:
%pip install -q -e . --no-deps
print('installed package without changing Kaggle JAX/libtpu')

## v5e-8 topology and conservative model defaults

In [ ]:
from nano_dsv41f import ModelConfig,TrainConfig,V5E,make_v5e_mesh,runtime_report,semantic_axes,validate_v5e_runtime,validate_sequence_length
config=ModelConfig(); train_config=TrainConfig(seq_len=4096)
print(runtime_report()); [print('WARNING:',w) for w in validate_v5e_runtime()]
mesh=make_v5e_mesh(); print('mesh:',mesh); print('v5e:',V5E); print('vocab_size:',config.vocab_size); print('axes:',semantic_axes(config,mesh)); [print('WARNING:',w) for w in validate_sequence_length(train_config.seq_len,config,mesh)]

## Direct-to-shard BF16 initialization

In [ ]:
from nano_dsv41f import init_model_sharded_mixed_precision,init_optimizer_state_sharded,memory_report,precision_summary
params,param_specs,param_shardings=init_model_sharded_mixed_precision(jax.random.PRNGKey(0),config,mesh,payload_dtype=jnp.bfloat16)
jax.block_until_ready(jax.tree_util.tree_leaves(params)[0]); print('dtypes:',precision_summary(params)); print('memory:',memory_report(params,param_specs,mesh))
opt_state,opt_state_shardings=init_optimizer_state_sharded(params,param_specs,config,mesh); jax.block_until_ready(jax.tree_util.tree_leaves(opt_state)[0])

## Compile static base / late-indexer executables

In [ ]:
from nano_dsv41f import compile_pretrain_step
base_step=compile_pretrain_step(params,opt_state,param_specs,config,train_config,mesh,include_indexer=False,n_segments=None)
late_indexer_step=compile_pretrain_step(params,opt_state,param_specs,config,train_config,mesh,include_indexer=True,n_segments=1)

## Synthetic `T=1024` smoke batch

In [ ]:
import numpy as np
from nano_dsv41f import put_training_batch
smoke_t=1024; h=np.arange(smoke_t,dtype=np.int32)[None,:]%config.vocab_size; s=np.zeros_like(h); m=np.ones_like(h,dtype=bool)
ids,segments,token_mask=put_training_batch(h,s,m,config,mesh); print('input sharding:',ids.sharding); print('local shard:',ids.addressable_shards[0].data.shape)

## Compile diagnostics before execution

In [ ]:
from nano_dsv41f import compile_diagnostics
zero_step=jnp.asarray(0,jnp.int32); compiled_base,diag=compile_diagnostics(base_step,params,opt_state,ids,segments,zero_step,token_mask)
print('collectives:',diag['collectives']); print('compiler memory:',diag['memory']); print({k:v for k,v in diag['cost'].items() if any(t in k.lower() for t in ('flop','byte','transcend'))})

## Standalone sharded Splash local-MQA parity

In [ ]:
from nano_dsv41f.splash import dense_local_mqa_reference,make_v5e_sharded_local_mqa
kq,kk=jax.random.split(jax.random.PRNGKey(7)); q=jax.random.normal(kq,(1,smoke_t,config.attention.n_heads,config.attention.head_dim),dtype=jnp.bfloat16); kv=jax.random.normal(kk,(1,smoke_t,config.attention.head_dim),dtype=jnp.bfloat16); seg=jnp.zeros((1,smoke_t),jnp.int32)
fn=make_v5e_sharded_local_mqa(mesh,seq_len=smoke_t,n_heads=config.attention.n_heads,head_dim=config.attention.head_dim,local_window=config.attention.local_window)
so,sl=jax.jit(fn)(q,kv,seg); ro,rl=jax.jit(lambda q,kv,s:dense_local_mqa_reference(q,kv,s,local_window=config.attention.local_window))(q,kv,seg); jax.block_until_ready(so)
print('output max error:',float(jnp.max(jnp.abs(so.astype(jnp.float32)-ro.astype(jnp.float32))))); print('LSE max error:',float(jnp.max(jnp.abs(sl-rl))))

## First training step

In [ ]:
params,opt_state,metrics=base_step(params,opt_state,ids,segments,zero_step,token_mask); jax.block_until_ready(metrics['loss']); print({k:float(v) for k,v in metrics.items() if getattr(v,'ndim',1)==0})

## Next

If the step is finite and HBM is comfortable, attach a pretrained ~32K tokenizer and packed data pipeline. Scale real tokens only after this smoke test; then use profiler evidence to choose between attention-kernel work and MoE dispatch work.